# Testing Pandas for EBD Processing

Note: this data is pulling from an unzipped txt EBD file in the "ebd" folder not included in the repository

## Resources

- [Pandas vs R Cheatsheet](https://datascientyst.com/pandas-vs-r-cheat-sheet/)
- [Combining/Merge/Axis with dataframes](https://www.geeksforgeeks.org/pandas/how-to-combine-two-dataframe-in-python-pandas/)

In [18]:
import pandas as pd
import json
import numpy as np
import geopandas as gpd


# eBird Dataset file name
ebd_fn = 'ebd_US-NC_202101_202602_unv_smp_relMar-2026.txt'

## Load the EBD into a Pandas DF

In [ ]:
# define data types
dtypes = {'LAST EDITED DATE' : str, 'COUNTRY' : str, 'COUNTRY CODE' : str,
'STATE' : str, 'STATE CODE' : str, 'COUNTY' : str, 'COUNTY CODE' : str,
'IBA CODE' : str, 'BCR CODE' : str, 'USFWS CODE' : str, 'ATLAS BLOCK' : str,
'LOCALITY' : str, 'LOCALITY ID' : str, 'LOCALITY TYPE' : str,
'LATITUDE' : float, 'LONGITUDE' : float, 'OBSERVATION DATE' : str,
'TIME OBSERVATIONS STARTED' : str, 'OBSERVER ID' : str, 
'OBSERVER ORCID ID' : str, 'SAMPLING EVENT IDENTIFIER' : str,
'OBSERVATION TYPE' : str, 'PROTOCOL NAME' : str, 'PROTOCOL CODE' : str,
'PROJECT NAMES' : object, 'PROJECT IDENTIFIERS' : object, 'DURATION MINUTES' : float,
'EFFORT DISTANCE KM' : str, 'EFFORT AREA HA' : str, 'NUMBER OBSERVERS' : float,
'ALL SPECIES REPORTED' : str, 'GROUP IDENTIFIER' : str, 'HAS MEDIA' : str,
'CHECKLIST COMMENTS' : str, 'GLOBAL UNIQUE IDENTIFIER' : str, 
'TAXONOMIC ORDER' : str, 'CATEGORY' : str, 'TAXON CONCEPT ID' : str,
'COMMON NAME' : str, 'SCIENTIFIC NAME' : str, 'SUBSPECIES COMMON NAME' : str,
'SUBSPECIES SCIENTIFIC NAME' : str, 'EXOTIC CODE' : str, 
'OBSERVATION COUNT' : str, 'BREEDING CODE' : str, 'BREEDING CATEGORY' : str,
'BEHAVIOR CODE' : str, 'AGE/SEX' : str, 'APPROVED' : str, 'REVIEWED' : str,
'REASON' : str, 'SPECIES COMMENTS' : str}

ebd_raw = pd.read_csv(f'./ebd/{ebd_fn}', sep='\t', dtype = dtypes)

## Create Test Dataset

In [6]:
# create subset for testing
## PRODUCTION
# ebd = ebd_raw
## END PRODUCTION

### TEST
ebd = pd.read_csv("ebd_test.csv", dtype = dtypes)

## save data
# ebd = pd.concat([ebd_raw.head(1000), ebd_raw.tail(1000)])
# ebd.to_csv("ebd_test.csv")
# ebd_bigger = pd.concat([ebd_raw.head(10000), ebd_raw.tail(10000)])
# ebd_bigger.to_csv("ebd_test_bigger.csv")

### END TEST

ebd = ebd.fillna('')


# File for test.json
out_json = open('test.json', 'w', encoding = 'utf-8-sig')


## Start Exploring

In [7]:
rows_columns = ebd.shape
print(f'{rows_columns[0]:,d} rows, {rows_columns[1]} columns found')
# ebd['SAMPLING EVENT IDENTIFIER'].sort_values()

# remove spaces from colum names
ebd.columns = ebd.columns.str.replace(' ', '_')

# get column names
ebd_columns =  ebd.columns

# define observation fields
observation_fields = pd.Series([
  "GLOBAL_UNIQUE_IDENTIFIER",
  "TAXONOMIC_ORDER",
  "CATEGORY",
  "TAXON_CONCEPT_ID",
  "COMMON_NAME",
  "SCIENTIFIC_NAME",
  "SUBSPECIES_COMMON_NAME",
  "SUBSPECIES_SCIENTIFIC_NAME",
  "EXOTIC_CODE",
  "OBSERVATION_COUNT",
  "BREEDING_CODE",
  "BREEDING_CATEGORY",
  "BEHAVIOR_CODE",
  "AGE/SEX",
  "APPROVED",
  "REVIEWED",
  "REASON",
  "SPECIES_COMMENTS"
])

# get checklist fields
checklist_fields = pd.Series(ebd_columns[~ebd_columns.isin(observation_fields)])

# add SAMPLING EVENT IDENTIFIER to observations, remove Unamed column from checklist_fields
observation_fields = pd.concat([observation_fields, pd.Series(['SAMPLING_EVENT_IDENTIFIER'])])
checklist_fields = checklist_fields[checklist_fields != 'Unnamed: 52']

print(f'checklist fields:\n{list(checklist_fields)}')
print(f'observation fields:\n{list(observation_fields)}')

2,000 rows, 54 columns found
checklist fields:
['Unnamed:_0', 'LAST_EDITED_DATE', 'COUNTRY', 'COUNTRY_CODE', 'STATE', 'STATE_CODE', 'COUNTY', 'COUNTY_CODE', 'IBA_CODE', 'BCR_CODE', 'USFWS_CODE', 'ATLAS_BLOCK', 'LOCALITY', 'LOCALITY_ID', 'LOCALITY_TYPE', 'LATITUDE', 'LONGITUDE', 'OBSERVATION_DATE', 'TIME_OBSERVATIONS_STARTED', 'OBSERVER_ID', 'OBSERVER_ORCID_ID', 'SAMPLING_EVENT_IDENTIFIER', 'OBSERVATION_TYPE', 'PROTOCOL_NAME', 'PROTOCOL_CODE', 'PROJECT_NAMES', 'PROJECT_IDENTIFIERS', 'DURATION_MINUTES', 'EFFORT_DISTANCE_KM', 'EFFORT_AREA_HA', 'NUMBER_OBSERVERS', 'ALL_SPECIES_REPORTED', 'GROUP_IDENTIFIER', 'HAS_MEDIA', 'CHECKLIST_COMMENTS', 'Unnamed:_52']
observation fields:
['GLOBAL_UNIQUE_IDENTIFIER', 'TAXONOMIC_ORDER', 'CATEGORY', 'TAXON_CONCEPT_ID', 'COMMON_NAME', 'SCIENTIFIC_NAME', 'SUBSPECIES_COMMON_NAME', 'SUBSPECIES_SCIENTIFIC_NAME', 'EXOTIC_CODE', 'OBSERVATION_COUNT', 'BREEDING_CODE', 'BREEDING_CATEGORY', 'BEHAVIOR_CODE', 'AGE/SEX', 'APPROVED', 'REVIEWED', 'REASON', 'SPECIES_COMM

## Nest Observations, create JSON

In [20]:
# get block layer

gpd_blocks = gpd.read_file("blocks.geojson")
gpd_blocks.shape

(5453, 41)

In [30]:
# subset ebd for checklists, lat/lon only
ebd_checklists = ebd.groupby([
    'SAMPLING_EVENT_IDENTIFIER',
    'LATITUDE',
    'LONGITUDE'
    ]).size().reset_index()

gpd_checklists = gpd.GeoDataFrame(
    ebd_checklists,
    geometry = gpd.points_from_xy(
        ebd_checklists.LONGITUDE,
        ebd_checklists.LATITUDE
    ),
    crs = "EPSG:4326"
)

# spatial join with blocks to get block info

checklist_blocks = gpd_checklists.sjoin(
    gpd_blocks, how = 'left')

checklist_blocks.head()

,SAMPLING_EVENT_IDENTIFIER,LATITUDE,LONGITUDE,0,geometry,index_right,FID,block_code,name,subnat2,...,ID_BLOCK,ID_BLOCK_CODE,ID_EBD_NAME,ID_NCBA_BLOCK,ID_OLD_ID,ID_S123_NOSPACES,ID_S123_NOSPACES_TEMP,ID_S123_SPACES,ID_S123_SPACES_TEMP,ID_WEB_BLOCKMAP
0,S297587800,35.772085,-82.183934,6,POINT (-82.18393 35.77209),639,0,35082G2SE,Celo SE,US-NC-111,...,CELO-SE,35082G2SE,Celo SE,CELO-SE,Celo-SE,Celo_SE,Celo_SE,Celo SE,Celo SE,Celo SE
1,S297593603,35.772486,-82.176471,4,POINT (-82.17647 35.77249),639,0,35082G2SE,Celo SE,US-NC-111,...,CELO-SE,35082G2SE,Celo SE,CELO-SE,Celo-SE,Celo_SE,Celo_SE,Celo SE,Celo SE,Celo SE
2,S297594414,35.771472,-82.173763,1,POINT (-82.17376 35.77147),639,0,35082G2SE,Celo SE,US-NC-111,...,CELO-SE,35082G2SE,Celo SE,CELO-SE,Celo-SE,Celo_SE,Celo_SE,Celo SE,Celo SE,Celo SE
3,S297643082,35.827522,-82.341800,11,POINT (-82.34180 35.82752),3429,0,35082G3CW,Mount Mitchell CW,US-NC-199,...,MOUNT MITCHELL-CW,35082G3CW,Mount Mitchell CW,MOUNT_MITCHELL-CW,Mount-Mitchell-CW,Mount_Mitchell_CW,Mount_Mitchell_CW,Mount Mitchell CW,Mount Mitchell CW,Mount Mitchell CW
4,S297692628,35.875300,-82.364486,5,POINT (-82.36449 35.87530),3471,0,35082H3SW,Burnsville SW,US-NC-199,...,BURNSVILLE-SW,35082H3SW,Burnsville SW,BURNSVILLE-SW,Burnsville-SW,Burnsville_SW,Burnsville_SW,Burnsville SW,Burnsville SW,Burnsville SW


## Tinker with Grouping by Checklist

In [34]:
# obs_only = ebd[observation_fields]
# check_only = ebd[checklist_fields]


# add fields
ebd['YEAR'] = ebd['OBSERVATION_DATE'].str[:4].astype(int)
ebd['MONTH'] = ebd['OBSERVATION_DATE'].str[5:7].astype(int)
ebd['PROJECT_CODE'] = np.where(
    ebd['PROJECT_NAMES'].str.contains('North Carolina Bird Atlas'),
    'EBIRD_ATL_NC',
    'EBIRD'
)
ebd['ID_NCBA_BLOCK'] = ""
ebd['ID_BLOCK_CODE'] = ""
ebd['PRIORITY_BLOCK'] = ""
ebd['NCBA_JULIAN_DAY'] = ""
ebd['NCBA_WEEK'] = ""
ebd['NCBA_SEASON'] = ""
ebd['NCBA_QUARTER'] = ""
ebd['NCBA_DATE_ADDED'] = "2026-04-27"
ebd['NCBA_EBD_VER'] = "Mar-2026"
ebd['NCBA_OBSERVER'] = ""


ebd_grouped = (
    # ebd.groupby(checklist_fields)
    ebd.groupby([
        'SAMPLING_EVENT_IDENTIFIER', 'LAST_EDITED_DATE', 'COUNTRY',
        'COUNTRY_CODE', 'STATE', 'STATE_CODE', 'COUNTY', 'COUNTY_CODE',
        'IBA_CODE', 'BCR_CODE', 'USFWS_CODE', 'ATLAS_BLOCK', 'LOCALITY',
        'LOCALITY_ID', 'LOCALITY_TYPE', 'LATITUDE', 'LONGITUDE',
        'OBSERVATION_DATE', 'TIME_OBSERVATIONS_STARTED', 'OBSERVER_ID',
        'PROTOCOL_NAME', 'PROTOCOL_CODE', 'PROJECT_CODE', 
        'PROJECT_NAMES', 
        'DURATION_MINUTES', 'EFFORT_DISTANCE_KM', 'EFFORT_AREA_HA',
        'NUMBER_OBSERVERS', 'ALL_SPECIES_REPORTED', 'GROUP_IDENTIFIER',
        'CHECKLIST_COMMENTS', 'YEAR', 'MONTH', 'ID_BLOCK_CODE', 'ID_NCBA_BLOCK',
        'PRIORITY_BLOCK', 'NCBA_JULIAN_DAY', 'NCBA_WEEK', 'NCBA_SEASON',
        'NCBA_DATE_ADDED', 'NCBA_EBD_VER', 'NCBA_OBSERVER'
        ])
    # ebd.groupby(['SAMPLING EVENT IDENTIFIER'])
    .apply(lambda x: x[list(observation_fields)].to_dict('records'))
    .reset_index()
    .rename(columns={0:'OBSERVATIONS'})
    # .to_dict(orient='records')
)
ebd_grouped.head()
# ebd_json = ebd_grouped.to_dict(orient='records')
# out_json.write(json.dumps(ebd_json, indent = 2))



C:\Users\skanderson\AppData\Local\Temp\1\ipykernel_9908\2977843961.py:42: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x[list(observation_fields)].to_dict('records'))


,SAMPLING_EVENT_IDENTIFIER,LAST_EDITED_DATE,COUNTRY,COUNTRY_CODE,STATE,STATE_CODE,COUNTY,COUNTY_CODE,IBA_CODE,BCR_CODE,...,ID_BLOCK_CODE,ID_NCBA_BLOCK,PRIORITY_BLOCK,NCBA_JULIAN_DAY,NCBA_WEEK,NCBA_SEASON,NCBA_DATE_ADDED,NCBA_EBD_VER,NCBA_OBSERVER,OBSERVATIONS
0,S297587800,2026-02-01 17:36:01.035857,United States,US,North Carolina,US-NC,Yancey,US-NC-199,,28,...,,,,,,,2026-04-27,Mar-2026,,[{'GLOBAL_UNIQUE_IDENTIFIER': 'URN:CornellLabO...
1,S297593603,2026-02-05 13:30:42.603676,United States,US,North Carolina,US-NC,Yancey,US-NC-199,,28,...,,,,,,,2026-04-27,Mar-2026,,[{'GLOBAL_UNIQUE_IDENTIFIER': 'URN:CornellLabO...
2,S297594414,2026-02-01 18:08:53.448845,United States,US,North Carolina,US-NC,Yancey,US-NC-199,,28,...,,,,,,,2026-04-27,Mar-2026,,[{'GLOBAL_UNIQUE_IDENTIFIER': 'URN:CornellLabO...
3,S297643082,2026-02-02 02:05:21.176707,United States,US,North Carolina,US-NC,Yancey,US-NC-199,US-NC_383,28,...,,,,,,,2026-04-27,Mar-2026,,[{'GLOBAL_UNIQUE_IDENTIFIER': 'URN:CornellLabO...
4,S297692628,2026-02-02 09:27:23.174982,United States,US,North Carolina,US-NC,Yancey,US-NC-199,,28,...,,,,,,,2026-04-27,Mar-2026,,[{'GLOBAL_UNIQUE_IDENTIFIER': 'URN:CornellLabO...


In [ ]:
# data = {'productLine': ['Alpha', 'Alpha', 'Beta'],
#         'color': ['Red', 'Blue', 'Green'],
#         'specs': [[{'partId': 'A1', 'weight': 10}, {'partId': 'A2', 'weight': 20}],
#                   [{'partId': 'B1', 'weight': 30}],
#                   [{'partId': 'C1', 'weight': 40}]]}

# df = pd.DataFrame(data)
# # print(json.dumps(df.to_json(orient='records'), indent=2))
# with open('test.json', 'w', encoding = "utf-8-sig") as f:
#     json.dump(df, f, ensure_ascii=False, indent = 2)
